[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tol/osint/blob/main/notebooks/pdf_to_table_converter.ipynb)

# PDF Table Extractor with Upload

## Overview
This notebook provides a comprehensive solution for extracting tables from PDF files using multiple extraction methods. It features an interactive UI for uploading PDFs and exporting extracted tables to Excel or CSV formats.

## Features
- **Multiple Extraction Methods**:
  - pdfplumber: Fast extraction for structured PDFs
  - Camelot: Lattice and stream mode table detection
  - Tesseract OCR: Handles scanned PDFs with text recognition
- **Cascading Fallback**: Automatically tries each method in sequence
- **Data Cleaning**: Removes empty rows/columns and handles NaN values
- **Multiple Export Formats**: Excel (with separate sheets) or CSV files
- **Interactive UI**: File upload, extraction, and export with progress indicators
- **Comprehensive Logging**: Track progress and debug issues

## Requirements
### System Dependencies
- **Tesseract OCR**: Install via Homebrew (macOS) - `brew install tesseract`
- **Ghostscript**: Install via Homebrew - `brew install ghostscript`

### Python Packages
- pdfplumber: PDF table extraction
- camelot-py: Advanced table detection with Ghostscript backend
- pytesseract: OCR text extraction
- pdf2image: PDF to image conversion
- openpyxl: Excel file creation
- ipywidgets: Interactive UI components
- pandas: Data manipulation
- numpy: Numerical operations

In [38]:
import subprocess
import sys

packages = [
    'pdfplumber',
    'camelot-py',
    'pytesseract',
    'pdf2image',
    'openpyxl',
    'ipywidgets',
    'pandas',
    'numpy'
]

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('✓ All dependencies installed successfully')

✓ All dependencies installed successfully


In [39]:
# Install system dependencies for pdf2image and Camelot
!apt-get update
!apt-get install -y poppler-utils ghostscript
print('✓ System dependencies (poppler, ghostscript) installed')

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ghostscript is already the newest version (9.55.0~dfsg1-0ubuntu5.13).
poppler-utils is already

In [40]:
import logging
import tempfile
import os
from pathlib import Path
from typing import List, Optional, Tuple
from datetime import datetime
import warnings

import pandas as pd
import numpy as np
import pdfplumber
import camelot
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
import openpyxl
from openpyxl.utils.dataframe import dataframe_to_rows
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('PDFTableExtractor')

print('✓ All libraries imported successfully')
print('✓ Logging configured')

✓ All libraries imported successfully
✓ Logging configured


In [41]:
import platform

system = platform.system()

if system == 'Darwin':
    pytesseract.pytesseract.pytesseract_cmd = '/usr/local/bin/tesseract'
    logger.info('Tesseract path configured for macOS')
elif system == 'Windows':
    pytesseract.pytesseract.pytesseract_cmd = r'C:\\Program Files\\Tesseract-OCR\\tesseract.exe'
    logger.info('Tesseract path configured for Windows')
elif system == 'Linux':
    pytesseract.pytesseract.pytesseract_cmd = '/usr/bin/tesseract'
    logger.info('Tesseract path configured for Linux')

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

print(f'✓ Tesseract configured for {system}')
print('✓ Warnings suppressed')

✓ Tesseract configured for Linux
✓ Warnings suppressed


In [42]:
def extract_with_pdfplumber(pdf_path: str) -> List[pd.DataFrame]:
    """Extract tables from PDF using pdfplumber."""
    dataframes = []
    try:
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f'PDF file not found: {pdf_path}')
        logger.info(f'Extracting tables with pdfplumber from {os.path.basename(pdf_path)}')
        with pdfplumber.open(pdf_path) as pdf:
            total_pages = len(pdf.pages)
            logger.info(f'PDF has {total_pages} pages')
            for page_num, page in enumerate(pdf.pages, 1):
                tables = page.extract_tables()
                if tables:
                    for table_idx, table in enumerate(tables, 1):
                        df = pd.DataFrame(table[1:], columns=table[0] if table else [])
                        dataframes.append(df)
                        logger.debug(f'Page {page_num}, Table {table_idx}: {df.shape[0]} rows, {df.shape[1]} columns')
        logger.info(f'✓ pdfplumber extracted {len(dataframes)} table(s)')
        return dataframes
    except FileNotFoundError as e:
        logger.error(f'File error: {str(e)}')
        raise
    except Exception as e:
        logger.warning(f'pdfplumber extraction failed: {str(e)}')
        return []

print('✓ pdfplumber extraction function defined')

✓ pdfplumber extraction function defined


In [43]:
def extract_with_camelot(pdf_path: str) -> List[pd.DataFrame]:
    """Extract tables from PDF using Camelot."""
    dataframes = []
    try:
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f'PDF file not found: {pdf_path}')
        logger.info(f'Extracting tables with Camelot (lattice mode) from {os.path.basename(pdf_path)}')
        try:
            tables = camelot.read_pdf(pdf_path, flavor='lattice', suppress_stdout=True)
            if len(tables) > 0:
                for table in tables:
                    df = table.df
                    dataframes.append(df)
                    logger.debug(f'Lattice: {df.shape[0]} rows, {df.shape[1]} columns')
                logger.info(f'✓ Camelot (lattice) extracted {len(dataframes)} table(s)')
                return dataframes
        except Exception as e:
            logger.debug(f'Lattice mode failed: {str(e)}')
        logger.info('Falling back to Camelot stream mode')
        tables = camelot.read_pdf(pdf_path, flavor='stream', suppress_stdout=True)
        if len(tables) > 0:
            for table in tables:
                df = table.df
                dataframes.append(df)
                logger.debug(f'Stream: {df.shape[0]} rows, {df.shape[1]} columns')
            logger.info(f'✓ Camelot (stream) extracted {len(dataframes)} table(s)')
            return dataframes
        logger.warning('Camelot extracted no tables')
        return []
    except FileNotFoundError as e:
        logger.error(f'File error: {str(e)}')
        raise
    except Exception as e:
        logger.warning(f'Camelot extraction failed: {str(e)}')
        return []

print('✓ Camelot extraction function defined')

✓ Camelot extraction function defined


In [44]:
def extract_with_tesseract(pdf_path: str) -> List[pd.DataFrame]:
    """Extract tables from PDF using Tesseract OCR."""
    dataframes = []
    try:
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f'PDF file not found: {pdf_path}')
        logger.info(f'Extracting with Tesseract OCR from {os.path.basename(pdf_path)}')
        try:
            images = convert_from_path(pdf_path, first_page=1, last_page=1)
            logger.info(f'Converted 1 page to image for OCR')
        except Exception as e:
            logger.warning(f'PDF to image conversion failed: {str(e)}')
            return []
        if images:
            image = images[0]
            try:
                text = pytesseract.image_to_string(image)
                lines = text.strip().split('\\n')
                if lines:
                    data = [{'text': line} for line in lines if line.strip()]
                    df = pd.DataFrame(data)
                    dataframes.append(df)
                    logger.info(f'✓ Tesseract extracted {len(data)} lines of text')
                    return dataframes
            except Exception as e:
                logger.warning(f'Tesseract OCR failed: {str(e)}')
        return []
    except FileNotFoundError as e:
        logger.error(f'File error: {str(e)}')
        raise
    except Exception as e:
        logger.warning(f'Tesseract extraction failed: {str(e)}')
        return []

print('✓ Tesseract extraction function defined')

✓ Tesseract extraction function defined


In [45]:
def extract_tables_cascading(pdf_path: str) -> List[pd.DataFrame]:
    """Extract tables using cascading methods."""
    try:
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f'PDF file not found: {pdf_path}')
        logger.info('=' * 60)
        logger.info(f'Starting cascading extraction for {os.path.basename(pdf_path)}')
        logger.info('=' * 60)
        logger.info('Attempt 1: pdfplumber')
        try:
            tables = extract_with_pdfplumber(pdf_path)
            if tables and len(tables) > 0:
                logger.info('✓ SUCCESS: pdfplumber extracted tables')
                return tables
        except Exception as e:
            logger.debug(f'pdfplumber failed: {str(e)}')
        logger.info('Attempt 2: Camelot')
        try:
            tables = extract_with_camelot(pdf_path)
            if tables and len(tables) > 0:
                logger.info('✓ SUCCESS: Camelot extracted tables')
                return tables
        except Exception as e:
            logger.debug(f'Camelot failed: {str(e)}')
        logger.info('Attempt 3: Tesseract OCR')
        try:
            tables = extract_with_tesseract(pdf_path)
            if tables and len(tables) > 0:
                logger.info('✓ SUCCESS: Tesseract extracted text')
                return tables
        except Exception as e:
            logger.debug(f'Tesseract failed: {str(e)}')
        logger.warning('✗ All extraction methods failed to extract tables')
        return []
    except FileNotFoundError as e:
        logger.error(f'File error: {str(e)}')
        raise
    except Exception as e:
        logger.error(f'Cascading extraction error: {str(e)}')
        return []

print('✓ Cascading extraction function defined')

✓ Cascading extraction function defined


In [46]:
def clean_and_validate_table(df: pd.DataFrame) -> pd.DataFrame:
    """Clean and validate a DataFrame from table extraction."""
    try:
        if df is None or df.empty:
            logger.warning('Empty DataFrame provided for cleaning')
            return pd.DataFrame()
        original_shape = df.shape
        df = df.dropna(how='all')
        df = df.dropna(axis=1, how='all')
        for col in df.columns:
            if df[col].dtype == 'object':
                df[col] = df[col].astype(str).str.strip()
        df = df.fillna(method='ffill')
        df = df.dropna(how='any')
        df = df.reset_index(drop=True)
        new_shape = df.shape
        logger.debug(f'Cleaned table: {original_shape} → {new_shape}')
        return df
    except Exception as e:
        logger.error(f'Error cleaning table: {str(e)}')
        return pd.DataFrame()

print('✓ Table cleaning function defined')

✓ Table cleaning function defined


In [47]:
def process_uploaded_pdf(uploaded_file_obj) -> List[pd.DataFrame]:
    """Process an uploaded PDF file and extract cleaned tables with robust widget handling."""
    try:
        if not uploaded_file_obj.value:
            raise ValueError('No file uploaded or file list is empty.')

        val = uploaded_file_obj.value

        # Extract file info from dict or list/tuple
        if isinstance(val, dict):
            file_info = list(val.values())[0]
        elif isinstance(val, (list, tuple)):
            file_info = val[0]
        else:
            raise ValueError(f'Unexpected widget value format: {type(val)}')

        # Robust filename extraction (handles metadata nesting)
        filename = file_info.get('name')
        if not filename and 'metadata' in file_info:
            filename = file_info['metadata'].get('name')

        if not filename:
            filename = 'uploaded_file.pdf'

        file_content = file_info.get('content')
        if file_content is None:
            raise KeyError('File content not found in upload data.')

        if not filename.lower().endswith('.pdf'):
            raise ValueError(f'Invalid file type: {filename}. Only PDF files are supported.')

        logger.info(f'Processing file: {filename}')

        with tempfile.NamedTemporaryFile(suffix='.pdf', delete=False) as tmp_file:
            tmp_file.write(file_content)
            tmp_path = tmp_file.name

        try:
            tables = extract_tables_cascading(tmp_path)
            if not tables:
                return []

            cleaned_tables = []
            for idx, table in enumerate(tables, 1):
                cleaned = clean_and_validate_table(table)
                if not cleaned.empty:
                    cleaned_tables.append(cleaned)

            return cleaned_tables
        finally:
            if os.path.exists(tmp_path):
                os.remove(tmp_path)
    except Exception as e:
        logger.error(f'Error processing uploaded PDF: {str(e)}')
        raise

print('✓ PDF processing function fixed to handle nested metadata')

✓ PDF processing function fixed to handle nested metadata


In [48]:
def export_to_excel(dataframes: List[pd.DataFrame], filename: str = 'extracted_tables.xlsx') -> str:
    """Export extracted tables to an Excel file with separate sheets."""
    try:
        if not dataframes:
            raise ValueError('No DataFrames to export')
        logger.info(f'Exporting {len(dataframes)} table(s) to Excel')
        output_path = filename
        with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
            for idx, df in enumerate(dataframes, 1):
                sheet_name = f'Table_{idx}'
                df.to_excel(writer, sheet_name=sheet_name, index=False)
                logger.debug(f'✓ Sheet \"{sheet_name}\" created with {df.shape[0]} rows')
        logger.info(f'✓ Excel file created: {output_path}')
        return output_path
    except ValueError as e:
        logger.error(f'Validation error: {str(e)}')
        raise
    except Exception as e:
        logger.error(f'Error exporting to Excel: {str(e)}')
        raise

print('✓ Excel export function defined')

✓ Excel export function defined


In [49]:
def export_to_csv(dataframes: List[pd.DataFrame], base_filename: str = 'extracted_table') -> List[str]:
    """Export extracted tables to separate CSV files."""
    try:
        if not dataframes:
            raise ValueError('No DataFrames to export')
        logger.info(f'Exporting {len(dataframes)} table(s) to CSV files')
        file_paths = []
        for idx, df in enumerate(dataframes, 1):
            filename = f'{base_filename}_{idx}.csv'
            df.to_csv(filename, index=False)
            file_paths.append(filename)
            logger.debug(f'✓ CSV file created: {filename} with {df.shape[0]} rows')
        logger.info(f'✓ {len(file_paths)} CSV file(s) created')
        return file_paths
    except ValueError as e:
        logger.error(f'Validation error: {str(e)}')
        raise
    except Exception as e:
        logger.error(f'Error exporting to CSV: {str(e)}')
        raise

print('✓ CSV export function defined')

✓ CSV export function defined


In [50]:
def create_download_button(filepath: str, button_label: str = 'Download') -> widgets.HTML:
    """Create a download link for a file."""
    try:
        if not os.path.exists(filepath):
            raise FileNotFoundError(f'File not found: {filepath}')
        filename = os.path.basename(filepath)
        file_size = os.path.getsize(filepath)
        file_size_kb = file_size / 1024
        html = f'<div style="margin: 10px 0;"><a href="files/{filepath}" download="{filename}" style="display: inline-block; padding: 8px 16px; background-color: #007bff; color: white; text-decoration: none; border-radius: 4px; font-weight: bold;">{button_label}: {filename} ({file_size_kb:.1f} KB)</a></div>'
        logger.debug(f'Download link created for {filename}')
        return widgets.HTML(html)
    except FileNotFoundError as e:
        logger.error(f'File error: {str(e)}')
        raise
    except Exception as e:
        logger.error(f'Error creating download button: {str(e)}')
        raise

print('✓ Download button function defined')

✓ Download button function defined


In [52]:
extracted_tables = []
file_upload = widgets.FileUpload(accept='.pdf', multiple=False, description='Upload PDF', button_style='info')
extract_button = widgets.Button(description='Extract Tables', button_style='success', tooltip='Extract tables from the uploaded PDF', icon='magic')
export_excel_button = widgets.Button(description='Export to Excel', button_style='warning', disabled=True, tooltip='Export extracted tables to Excel', icon='download')
export_csv_button = widgets.Button(description='Export to CSV', button_style='warning', disabled=True, tooltip='Export extracted tables to CSV', icon='download')
clear_button = widgets.Button(description='Clear All', button_style='danger', tooltip='Clear all uploads and extracted tables', icon='trash')
output_area = widgets.Output()
status_label = widgets.HTML('<p style="color: #666; font-style: italic;">Ready to process PDFs</p>')

def on_extract_clicked(button):
    global extracted_tables
    with output_area:
        clear_output()
        try:
            print('🔄 Processing PDF...')
            extracted_tables = process_uploaded_pdf(file_upload)
            if not extracted_tables:
                print('❌ Error: No tables were extracted from the PDF.')
                status_label.value = '<p style="color: #dc3545;">❌ Extraction failed</p>'
                return
            print(f'✓ Successfully extracted {len(extracted_tables)} table(s)\n')
            for idx, df in enumerate(extracted_tables, 1):
                print(f'Table {idx}: {df.shape[0]} rows × {df.shape[1]} columns')
                print(df.head(3).to_string())
                print()
            export_excel_button.disabled = False
            export_csv_button.disabled = False
            status_label.value = '<p style="color: #28a745;">✓ Tables extracted successfully</p>'
        except Exception as e:
            print(f'❌ Error: {str(e)}')
            status_label.value = f'<p style="color: #dc3545;">❌ Error: {str(e)}</p>'

def on_export_excel_clicked(button):
    with output_area:
        clear_output()
        try:
            if not extracted_tables:
                print('❌ No tables to export.')
                return
            filepath = export_to_excel(extracted_tables)
            display(create_download_button(filepath, 'Download Excel'))
        except Exception as e:
            print(f'❌ Export Error: {str(e)}')

def on_export_csv_clicked(button):
    with output_area:
        clear_output()
        try:
            if not extracted_tables:
                print('❌ No tables to export.')
                return
            filepaths = export_to_csv(extracted_tables)
            for filepath in filepaths:
                display(create_download_button(filepath, 'Download CSV'))
        except Exception as e:
            print(f'❌ Export Error: {str(e)}')

def on_clear_clicked(button):
    global extracted_tables
    with output_area:
        clear_output()
        extracted_tables = []
        file_upload.value.clear()
        export_excel_button.disabled = True
        export_csv_button.disabled = True
        status_label.value = '<p style="color: #666; font-style: italic;">Ready to process PDFs</p>'

extract_button.on_click(on_extract_clicked)
export_excel_button.on_click(on_export_excel_clicked)
export_csv_button.on_click(on_export_csv_clicked)
clear_button.on_click(on_clear_clicked)

title = widgets.HTML('<h2 style="color: #333;">📄 PDF Table Extractor</h2>')
upload_section = widgets.VBox([widgets.HTML('<h4>Step 1: Upload PDF</h4>'), file_upload])
buttons_section = widgets.VBox([widgets.HTML('<h4>Step 2: Process & Export</h4>'), widgets.HBox([extract_button, export_excel_button, export_csv_button, clear_button])])
results_section = widgets.VBox([widgets.HTML('<h4>Status & Results:</h4>'), status_label, output_area])
main_container = widgets.VBox([title, upload_section, buttons_section, results_section])
display(main_container)
print('\n✓ UI re-initialized with robust error handling')


✓ UI re-initialized with robust error handling
